# XGBoost Tutorial (Colab — Kaggle 다운로드 버전)

Kaggle의 [XGBoost Tutorial](https://www.kaggle.com/code/dansbecker/xgboost) (by Dan Becker) 을 **Google Colab에서 실행**하는 노트북입니다.

- 실행 시 **Kaggle API로 대회 데이터를 직접 다운로드**합니다 ([House Prices](https://www.kaggle.com/c/house-prices-advanced-regression-techniques) 의 `train.csv`).
- Kaggle API 토큰이 **노트북에 내장**되어 있어 별도 입력 없이 바로 실행됩니다.
- 원본은 2018년 코드라 일부 구식 API를 최신 라이브러리에 맞게 갱신했습니다:
  - `sklearn.preprocessing.Imputer` → `sklearn.impute.SimpleImputer`
  - `DataFrame.as_matrix()` → `.to_numpy()`
  - `fit(early_stopping_rounds=...)` → 생성자 인자 `XGBRegressor(early_stopping_rounds=...)` (xgboost 2.x+)

> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부에 공유/업로드하지 마세요.  
> (토큰 재발급/폐기: https://www.kaggle.com/settings/api)

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
# 필요한 라이브러리 설치
import sys, subprocess
for pkg in ["kaggle", "xgboost", "numpy", "pandas", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os

# Kaggle API 토큰 (내장)
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"

print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 House Prices 데이터 다운로드

In [ ]:
import os, glob, zipfile

WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()

# 대회 데이터 다운로드 + 압축 해제
api.competition_download_files("house-prices-advanced-regression-techniques", path=WORK_DIR, quiet=False)
for zpath in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(WORK_DIR)
    os.remove(zpath)

print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])

## 2. 데이터 준비

`train_X`, `test_X`, `train_y`, `test_y` 로 데이터를 나누고 결측치를 채웁니다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer  # (원본의 Imputer 대체)

data = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
data.dropna(axis=0, subset=["SalePrice"], inplace=True)
y = data.SalePrice
X = data.drop(["SalePrice"], axis=1).select_dtypes(exclude=["object"])
train_X, test_X, train_y, test_y = train_test_split(
    X.to_numpy(), y.to_numpy(), test_size=0.25, random_state=0
)

my_imputer = SimpleImputer()
train_X = my_imputer.fit_transform(train_X)
test_X = my_imputer.transform(test_X)
print("train:", train_X.shape, "| test:", test_X.shape)

## 3. 모델 학습

scikit-learn 과 동일한 방식으로 모델을 만들고 학습합니다.

In [ ]:
from xgboost import XGBRegressor

my_model = XGBRegressor()
my_model.fit(train_X, train_y, verbose=False)

## 4. 평가 및 예측

In [ ]:
# 예측
predictions = my_model.predict(test_X)

from sklearn.metrics import mean_absolute_error
print("Mean Absolute Error : " + str(mean_absolute_error(predictions, test_y)))

## 5. 모델 튜닝

### n_estimators & early_stopping_rounds
- **n_estimators**: 모델링 사이클 반복 횟수. 너무 작으면 과소적합, 너무 크면 과대적합.
- **early_stopping_rounds**: 검증 점수가 개선되지 않으면 자동으로 학습을 멈춤.

> 최신 xgboost에서는 `early_stopping_rounds` 를 **생성자**에 전달합니다.

In [ ]:
my_model = XGBRegressor(n_estimators=1000, early_stopping_rounds=5)
my_model.fit(train_X, train_y, eval_set=[(test_X, test_y)], verbose=False)
print("best_iteration:", my_model.best_iteration)

### learning_rate

각 트리의 기여도를 작게(`learning_rate`) 하면서 트리 수를 늘리면 보통 더 정확한 모델을 얻습니다.

In [ ]:
my_model = XGBRegressor(n_estimators=1000, learning_rate=0.05, early_stopping_rounds=5)
my_model.fit(train_X, train_y, eval_set=[(test_X, test_y)], verbose=False)

predictions = my_model.predict(test_X)
print("Mean Absolute Error : " + str(mean_absolute_error(predictions, test_y)))
print("best_iteration:", my_model.best_iteration)

### n_jobs

큰 데이터셋에서 학습 속도가 중요하다면 `n_jobs` 를 코어 수에 맞춰 병렬 처리할 수 있습니다 (결과 품질은 동일, 작은 데이터셋에선 효과 미미).

In [ ]:
my_model = XGBRegressor(n_estimators=1000, learning_rate=0.05, n_jobs=4, early_stopping_rounds=5)
my_model.fit(train_X, train_y, eval_set=[(test_X, test_y)], verbose=False)

predictions = my_model.predict(test_X)
print("Mean Absolute Error : " + str(mean_absolute_error(predictions, test_y)))

## 결론

**XGBoost** 는 표준 정형 데이터(테이블 형태)에서 널리 쓰이는 강력한 그래디언트 부스팅 라이브러리입니다. `n_estimators`, `early_stopping_rounds`, `learning_rate`, `n_jobs` 등의 파라미터를 조정해 정확도와 속도를 최적화하세요.

## 6. 제출 파일 생성 & 내려받기

학습된 `my_model` 로 **실제 테스트셋(`test.csv`)** 을 예측해 `submission.csv`(`Id`, `SalePrice`)를 만들고 내려받습니다. (학습 때와 동일한 수치형 컬럼·`my_imputer` 결측 보정을 적용합니다.)

In [ ]:
# 실제 테스트셋(test.csv) 예측 → submission.csv 생성
test_full = pd.read_csv(os.path.join(WORK_DIR, "test.csv"))
X_test_real = my_imputer.transform(test_full[X.columns].to_numpy())  # 학습과 동일 컬럼·imputer
test_preds = my_model.predict(X_test_real)

submission_path = os.path.join(WORK_DIR, "submission.csv")
pd.DataFrame({"Id": test_full.Id, "SalePrice": test_preds}).to_csv(submission_path, index=False)
print("Saved:", submission_path)

# 내려받기
try:
    from google.colab import files
    files.download(submission_path)
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기

위에서 생성된 `submission.csv` 를 아래 페이지에서 업로드해 제출하세요:

👉 **[House Prices 제출 페이지](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/submit)**

- 대회 홈: https://www.kaggle.com/c/house-prices-advanced-regression-techniques
> 제출하려면 해당 대회에 Join(규칙 동의)되어 있어야 합니다.